In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
# from consensus_aligner import ChromatographicAligner
from ipyfilechooser import FileChooser
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess, sys, threading
import time



In [ ]:
class ChromatographicAlignerUI(Interface):
    def __init__(self):
        super().__init__(supported_extensions=('.txt',))
        self._setup_default_parameters()
        self._create_parameter_widgets()
        self._create_base_widgets()
        # self._create_action_widgets() # déjà créé dans _create_alignment_widget
        self._create_alignment_widget()
        self._setup_callbacks()
        self._setup_environment()
        self._create_optional_filter_widgets()

        # états
        self.align_state = "idle"         # "idle" | "running" | "ready" | "error"
        self.has_alignment_results = False

    def _setup_default_parameters(self):
        """Set up default parameters for the chromatographic alignment."""
        #public configurable
        self.rt1_penalty = "1"
        self.rt2_penalty = "5"
        self.similarity_cutoff = "90"
        self.missing_value_limit = 0.05 # MON FILTER

        # private, fixed
        self._disimilarity_cutoff = 90
        self._num_cores = 1
        self._quant_method= "T"
        self._auto_tune_match_stringency = False
        self._missing_peak_finder_similarity_lax = 0.85
        self._match_factor_min = 650

    
    def _create_parameter_widgets(self):
        self.w_seedFile = widgets.Text(value='1')
        self.seedFile = self._bold_widget("Seed file", self.w_seedFile)
        self.seed_def = self.create_help_text(
             "File number in inputFileList to initialize alignment."
        )
        self.w_rt1_penalty = widgets.Text(value=self.rt1_penalty)
        self.rt1_penalty = self._bold_widget("RT1 Penalty", self.w_rt1_penalty)
        self.rt1_penalty_def = self.create_help_text(
            "Penalty used for first retention time errors.  Defaults to 1."
        )
        self.w_rt2_penalty = widgets.Text(value=self.rt2_penalty)
        self.rt2_penalty = self._bold_widget("RT2 Penalty", self.w_rt2_penalty)
        self.rt2_penalty_def = self.create_help_text(
            "Penalty used for second retention time errors. Defaults to 5."
        )
        self.w_similarity_cutoff = widgets.Text(value=self.similarity_cutoff)
        self.similarity_cutoff = self._bold_widget("Similarity Cutoff", self.w_similarity_cutoff)
        self.similarity_cutoff_def = self.create_help_text(
            "Adjusts peak similarity threshold required for alignment."
            "Adjust in concordance with RT1 and RT2 penalties. " \
            "Defaults to 90."
            )
        
    def _create_alignment_widget(self):
        self.txt_title = widgets.HTML(value="<H1>Chromatographic Alignment</H1>")
         # NIST matching
        self.nist = widgets.Checkbox(
            value=True,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        )
              # Action widgets
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()


    def _on_button_click(self, b):
        """Handle button click event."""

        self.output.append_stdout("Running alignment... ")
        # validate parameter #TODO
        # errors = self._validate_parameters()
        if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
            self.output.append_stdout("Output directory cannot be empty")
            return

        self.output.append_stdout("\n Collecting files from selections...   ")
        selected_files = self.get_all_files_from_selections()
        if not selected_files:
            self.output.append_stdout("Please select files or folders containing .txt files.")
            return
        self.output.append_stdout(f"\n✅ {len(selected_files)} compatible files found")
        for i, f in enumerate(selected_files, 1):
            if f.startswith(self.docker_volume_path):
                display_path = f.replace(self.docker_volume_path, '')
            else:
                display_path = f
            self.output.append_stdout(f"\n  {i}. {display_path}")

        seed_file= int(self.w_seedFile.value.strip()) -1
        self.align_state = "running"
        self.has_alignment_results = False
            # ✅ DÉSACTIVER LE BOUTON FILTER PENDANT L'ALIGNMENT
        self.apply_filter_button.disabled = True
        self.apply_filter_button.tooltip = "Alignment in progress... please wait"
        
        self._start_subprocess_alignment(selected_files, str(seed_file))


    def _start_subprocess_alignment(self, selected_files, seed_file):
        """Start the alignment process in a subprocess."""
        self.output.append_stdout("\n" + "="*60 + "\n")
        self.output.append_stdout("🔄 Starting Peak Alignment ...\n") 

        alignment_params = [
            sys.executable,
            '/app/src/peak_alignment_cli.py',
            '--seed_file', seed_file,
            '--output_path', self.get_output_path(),
            '--rt1_penalty', self.w_rt1_penalty.value,
            '--rt2_penalty', self.w_rt2_penalty.value,
            '--similarity_cutoff', self.w_similarity_cutoff.value,
            '--disimilarity_cutoff', (self._disimilarity_cutoff),
            '--num_cores', (self._num_cores),
            '--missing_value_limit', (self.missing_value_limit),
            '--quant_method', self._quant_method,
            '--missing_peak_finder_similarity_lax', (self._missing_peak_finder_similarity_lax)
        ]
        # cas des booleens
        if self._auto_tune_match_stringency:
            alignment_params.append('--auto_tune_match_stringency')
        if self.nist.value:
            alignment_params.append('--nist')
        # cas des listes
        alignment_params +=["--input"] + selected_files
        
        alignment_params= list(map(str, alignment_params))

        start_time = time.time()
        self.current_process = subprocess.Popen(
            alignment_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env={'PYTHONUNBUFFERED': '1'}
        )
        try:
            while self.current_process.poll() is None and (time.time() - start_time) < 600:
                line = self.current_process.stdout.readline()
                if line:
                    self.output.append_stdout(line)
                    start_time = time.time()  # Reset timeout
                time.sleep(0.1)
            
            # Gestion de fin
            if self.current_process and self.current_process.poll() is None:
                # Timeout
                self.output.append_stdout("⏰ Analysis timed out\n")
                self.current_process.terminate()
                time.sleep(1)
                if self.current_process.poll() is None:
                    self.current_process.kill()
                self.align_state = "error"
                self.has_alignment_results = False
                # ✅ GARDER DÉSACTIVÉ en cas d'erreur
                self.apply_filter_button.disabled = True
                
            elif self.current_process:
                retcode = self.current_process.returncode
                if retcode == 0:
                    self.output.append_stdout("\n✅ Analysis completed successfully\n")
                    self.align_state = "ready"
                    self.has_alignment_results = True
                    
                    # ✅ DÉBLOQUER LE BOUTON
                    self.apply_filter_button.disabled = False
                    self.apply_filter_button.tooltip = "Click to apply post-processing filter"
                    
                    
                else:
                    self.output.append_stdout(f"\n❌ Analysis failed with code {retcode}\n")
                    self.align_state = "error"
                    self.has_alignment_results = False
                    # ✅ GARDER DÉSACTIVÉ en cas d'échec
                    self.apply_filter_button.disabled = True

        except Exception as e:
            self.output.append_stdout(f"❌ Error: {e}\n")
            self.align_state = "error"
            self.has_alignment_results = False
            # ✅ GARDER DÉSACTIVÉ en cas d'exception
            self.apply_filter_button.disabled = True
        finally:
            self.current_process = None
                
            
    def _create_optional_filter_widgets(self):
        self.w_new_missing_value_limit = widgets.Text(value="0.5")
        self.new_missing_value_limit = self._bold_widget("Missing Value Limit", self.w_new_missing_value_limit)
        self.missing_value_limit_def = self.create_help_text(
            "Maximum fraction (Numeric between 0 and 1) of missing values acceptable \
                for retaining a metabolite in the final alignment table. Defaults to 0.05 in \
                    chromatographic alignment. Defaults to 0.5 in optional post-processing."
        )
        self.apply_filter_button = widgets.Button(
            description='Apply Filter',
            button_style='info',
            icon='filter',
            style=self.style,
            disabled=True,
            tooltip="Run alignment first to enable this button"
        )
        self.apply_filter_button.on_click(self._on_filter_button_click)
        # print(f"🔍 Filter button created with {len(self.apply_filter_button._click_handlers)} handlers")
    
    
    def _on_filter_button_click(self, b):
        """Handle filter button click event."""
        # ✅ PAS BESOIN DE VÉRIFIER L'ÉTAT - le bouton n'est activé que si prêt
        self.output.append_stdout("✅ Starting filtering...\n")
        self.output.append_stdout(f"{'='*60}\n")

        outputpath = self.get_output_path()
        all_files = os.listdir(outputpath)
        all_files = [os.path.join(outputpath, f) for f in all_files]
        
        self._start_subprocess_alignment_filter(all_files)
        

    def _start_subprocess_alignment_filter(self, selected_files):
        """Start the filter process with cleanable output."""
        self.output.append_stdout("\n" + "="*60 + "\n")
        self.output.append_stdout("🔄 Starting Peak Alignment Filter ...\n") 
        
        filter_params = [
            sys.executable, '-u',
            '/app/src/peak_alignment_filter_cli.py',
            '--new_missing_value_limit', str(self.w_new_missing_value_limit.value),
            '--output_path', self.get_output_path(),
        ]
        filter_params += ['--path'] + selected_files
        filter_params = list(map(str, filter_params))

        start_time = time.time()
        self.current_process_filter = subprocess.Popen(
            filter_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env={'PYTHONUNBUFFERED': '1'}
        )
        try:
            while self.current_process_filter.poll() is None and (time.time() - start_time) < 600:
                line = self.current_process_filter.stdout.readline()
                if line:
                    self.output.append_stdout(line)
                    start_time = time.time()  # Reset timeout
                time.sleep(0.1)
            
                # Gestion de fin
            if self.current_process_filter and self.current_process_filter.poll() is None:
                self.output.append_stdout("⏰ Filter timed out\n")
                self.current_process_filter.terminate()
                time.sleep(1)
                if self.current_process_filter.poll() is None:
                    self.current_process_filter.kill()
            elif self.current_process_filter:
                retcode = self.current_process_filter.returncode
                if retcode == 0:
                    self.output.append_stdout("\n✅ Filter completed successfully\n")
                else:
                    self.output.append_stdout(f"\n❌ Filter failed with code {retcode}\n")

        except Exception as e:
            self.output.append_stdout(f"❌ Filter Error: {e}\n")
        finally:
            self.current_process_filter = None


    def display(self):
        """Display the interface."""
        display(self.txt_title,
                widgets.VBox([self._vbox, self._vbox2]),
                self.seedFile,
                self.seed_def,
                self.rt1_penalty,
                self.rt1_penalty_def,
                self.rt2_penalty,
                self.rt2_penalty_def,
                self.similarity_cutoff,
                self.similarity_cutoff_def,
                self.nist,
                widgets.HBox([self.run_button, self.stop_button, self.clear_button]),

                widgets.HTML("<hr><b>Optional Post-processing</b>"),
                self.new_missing_value_limit,
                self.missing_value_limit_def,
                self.apply_filter_button,
                widgets.HTML("<hr>"),
                self.output,
                )
    


In [ ]:
t = ChromatographicAlignerUI()
t.display()